
# GVH Diagonal Cubic 0.3.2.7.3.7.2.3 — Exact Canonical Hamiltonian and Constraint Densities from Total Kinetic Inverse

**Auteur :** Charlemagne O Laurince

## Mission
Partir de la branche générique non dégénérée de
\[
Q_{\rm total}=Q_{\rm EH}+Q_u
\]
validée en `0.3.2.7.3.7.2.2`, construire la transformée de Legendre et expliciter la partie cinétique de
\[
\mathcal C_\perp,\qquad \mathcal C_i.
\]

Le notebook ne fabrique pas les termes spatiaux ou de shift encore absents.

\[
\boxed{\mathrm{DISPERSION\_READY=False}}
\]


In [1]:

import sympy as sp, json
from pathlib import Path
print("GVH 0.3.2.7.3.7.2.3")
print("SymPy:", sp.__version__)


GVH 0.3.2.7.3.7.2.3
SymPy: 1.14.0



## 1. Base cinétique et témoin total exact

On utilise la même base cinétique que 7.2.2 :
\[
V^A=(K_{11},K_{22},K_{33},K_{12},K_{13},K_{23},S,W_1,W_2,W_3).
\]

Pour garder ce notebook léger et auditable, la transformée de Legendre est vérifiée sur le même témoin rationnel exact de branche non dégénérée.


In [2]:

# Build the exact rational 10x10 witness used as canonical branch representative.
# Symmetric, nonsingular, with metric-directional coupling retained.
Q = sp.Matrix([
[ 4, -1, -1, 0,0,0, 1, 1,0,0],
[-1,  4, -1, 0,0,0, 1, 0,1,0],
[-1, -1,  4, 0,0,0, 1, 0,0,1],
[ 0,  0,  0, 3,0,0, 0, 1,0,0],
[ 0,  0,  0, 0,3,0, 0, 0,1,0],
[ 0,  0,  0, 0,0,3, 0, 0,0,1],
[ 1,  1,  1, 0,0,0, 5, 0,0,0],
[ 1,  0,  0, 1,0,0, 0, 6,0,0],
[ 0,  1,  0, 0,1,0, 0, 0,6,0],
[ 0,  0,  1, 0,0,1, 0, 0,0,6],
])
assert Q == Q.T
assert Q.rank()==10
assert Q.det()!=0
print("Q_total witness rank =",Q.rank())


Q_total witness rank = 10



## 2. Transformée de Legendre exacte

Pour un secteur quadratique
\[
\mathcal L_{\rm kin}=\frac12V^TQV,
\]
les moments sont
\[
P=QV,
\]
et, si \(Q\) est inversible,
\[
V=Q^{-1}P.
\]

Le Hamiltonien cinétique canonique est alors
\[
\boxed{\mathcal H_{\rm kin}=\frac12P^TQ^{-1}P}.
\]


In [3]:

P = sp.Matrix(sp.symbols("P0:10", real=True))
Qinv = Q.inv()
Vsol = Qinv*P

L_on_shell = sp.Rational(1,2)*(Vsol.T*Q*Vsol)[0]
H_legendre = (P.T*Vsol)[0] - L_on_shell
H_expected = sp.Rational(1,2)*(P.T*Qinv*P)[0]

assert sp.expand(H_legendre-H_expected)==0
print("Legendre identity: PASS")
print("solved velocities =",len(Vsol))


Legendre identity: PASS
solved velocities = 10



## 3. Structure canonique avec lapse et shift

On écrit structurellement
\[
H_C
=
N\left(\mathcal H_{\rm kin}+\mathcal V\right)
+
N^i\mathcal C_i
+
\lambda_{\rm mult}\chi.
\]

Ici :
- \(\mathcal H_{\rm kin}\) est explicite ;
- \(\mathcal V\) représente encore les termes spatiaux à reconstruire ;
- \(\mathcal C_i\) restent à dériver après restauration complète du shift.


In [4]:

N,N1,N2,N3,lam = sp.symbols("N N1 N2 N3 lambda_mult", real=True)
Hk,Vsp,C1,C2,C3,chi = sp.symbols("Hk Vsp C1 C2 C3 chi")

HC_struct = N*(Hk+Vsp) + N1*C1 + N2*C2 + N3*C3 + lam*chi

Cperp_struct = sp.diff(HC_struct,N)
Ci_struct = [sp.diff(HC_struct,x) for x in (N1,N2,N3)]

assert Cperp_struct == Hk+Vsp
assert Ci_struct == [C1,C2,C3]
print("C_perp structural =",Cperp_struct)
print("C_i structural =",Ci_struct)


C_perp structural = Hk + Vsp
C_i structural = [C1, C2, C3]



## 4. Partie cinétique explicite de \(\mathcal C_\perp\)

Sur la branche générique :
\[
\boxed{
\mathcal C_{\perp,\rm kin}
=
\frac12P^TQ_{\rm total}^{-1}P.
}
\]

C'est la partie que 7.2.3 ferme réellement.


In [5]:

Cperp_kin = sp.expand(H_expected)
assert Cperp_kin.has(*P)
print("C_perp kinetic quadratic form derived: PASS")


C_perp kinetic quadratic form derived: PASS



## 5. Contraintes primaires et secondaires

On conserve
\[
p_N\approx0,\qquad p_{N^i}\approx0.
\]

Leur préservation implique structurellement
\[
\dot p_N=-\mathcal C_\perp\approx0,
\qquad
\dot p_{N^i}=-\mathcal C_i\approx0.
\]

Mais la forme **full-field** de \(\mathcal C_i\) n'est pas encore calculée.


In [6]:

constraint_map = {
    "p_N":"C_perp",
    "p_N1":"C_1",
    "p_N2":"C_2",
    "p_N3":"C_3",
}
constraint_map


{'p_N': 'C_perp', 'p_N1': 'C_1', 'p_N2': 'C_2', 'p_N3': 'C_3'}


## 6. Audit de \(R_{DD2}\)

La transformée de Legendre cinétique est désormais fermée sur la branche témoin exacte.

Cependant :
\[
\mathcal V,\quad \mathcal C_i
\]
ne sont pas encore reconstruits avec
\[
D_is,\ D_iv_j,\ a_i^{(n)},\ {}^{(3)}R,\ N^i.
\]

Donc :
\[
\boxed{R_{DD2}:\ \text{OPEN / UNCOMPUTED}}
\]

et non \(R_{DD2}=0\).


In [7]:

GATES = {
    "generic_total_inverse_available":True,
    "Legendre_transform_exact_on_witness":True,
    "Cperp_kinetic_explicit":True,
    "lapse_shift_constraint_structure_registered":True,
    "primary_secondary_map_registered":True,
    "full_spatial_potential_explicit":False,
    "full_Cperp_explicit":False,
    "full_Ci_explicit":False,
    "RDD2_computed":False,
    "hypersurface_algebra_closed":False,
}
for k,v in GATES.items():
    print(k,":",v)

FINAL_STATUS = (
    "PARTIAL-PASS-EXACT-CANONICAL-KINETIC-HAMILTONIAN_"
    "C-PERP-KINETIC-EXPLICIT_"
    "BLOCKED-SPATIAL-SHIFT-RECONSTRUCTION-AND-FULL-CONSTRAINT-DENSITIES"
)
DISPERSION_READY=False
assert DISPERSION_READY is False
print("\nFINAL STATUS:",FINAL_STATUS)
print("DISPERSION_READY =",DISPERSION_READY)


generic_total_inverse_available : True
Legendre_transform_exact_on_witness : True
Cperp_kinetic_explicit : True
lapse_shift_constraint_structure_registered : True
primary_secondary_map_registered : True
full_spatial_potential_explicit : False
full_Cperp_explicit : False
full_Ci_explicit : False
RDD2_computed : False
hypersurface_algebra_closed : False

FINAL STATUS: PARTIAL-PASS-EXACT-CANONICAL-KINETIC-HAMILTONIAN_C-PERP-KINETIC-EXPLICIT_BLOCKED-SPATIAL-SHIFT-RECONSTRUCTION-AND-FULL-CONSTRAINT-DENSITIES
DISPERSION_READY = False



## 7. Prochaine étape

### `0.3.2.7.3.7.2.4 — Spatial-Gradient and Shift Reconstruction of Exact Canonical Constraint Densities`

Elle devra réintroduire explicitement
\[
\mathcal D_\perp=\frac1N(\partial_t-\mathcal L_{\vec N}),
\]
ainsi que
\[
D_is,\quad D_iv_j,\quad a_i^{(n)},\quad {}^{(3)}R,
\]
puis dériver
\[
\mathcal C_\perp,\quad \mathcal C_i
\]
sous forme canonique complète.


In [8]:

artifact = {
    "notebook":"GVH_Diagonal_Cubic_0.3.2.7.3.7.2.3",
    "final_status":FINAL_STATUS,
    "Q_total_witness_rank":int(Q.rank()),
    "Q_total_witness_det_nonzero":bool(Q.det()!=0),
    "legendre_transform_pass":True,
    "Cperp_kinetic_formula":"1/2 P^T Q_total^{-1} P",
    "Cperp_full_status":"OPEN",
    "Ci_full_status":"OPEN",
    "RDD2_status":"OPEN_UNCOMPUTED",
    "gates":GATES,
    "dispersion_ready":False,
    "next":"GVH_Diagonal_Cubic_0.3.2.7.3.7.2.4_Spatial_Gradient_and_Shift_Reconstruction_of_Exact_Canonical_Constraint_Densities.ipynb"
}
export_dir = Path("/content/gvh_exports") if Path("/content").exists() else Path.cwd()/"gvh_exports"
export_dir.mkdir(parents=True,exist_ok=True)
artifact_path = export_dir/"gvh_0.3.2.7.3.7.2.3_canonical_hamiltonian_constraints.json"
artifact_path.write_text(json.dumps(artifact,indent=2),encoding="utf-8")
print("Artifact:",artifact_path)


Artifact: /content/gvh_exports/gvh_0.3.2.7.3.7.2.3_canonical_hamiltonian_constraints.json



# Conclusion

Cette étape ferme la transformée de Legendre cinétique sur une branche totale non dégénérée :

\[
\boxed{\mathcal H_{\rm kin}=\frac12P^TQ_{\rm total}^{-1}P}.
\]

Elle donne donc explicitement la partie cinétique de
\[
\mathcal C_\perp.
\]

Mais la reconstruction complète du potentiel spatial et du shift reste ouverte.

Verdict :
\[
\boxed{\text{PARTIAL PASS}}
\]

avec
\[
\boxed{\mathrm{DISPERSION\_READY=False}}.
\]
